# BellAI - Plataforma Modular de Entrenamiento en Google Colab

Este cuaderno esta dividido en celdas independientes:
- **Parte 1**: Instalacion y Configuracion (Se ejecuta solo una vez al inicio de sesion).
- **Partes 2 a 6**: Celdas dedicadas para cada uno de los 5 modelos de BellAI con formularios visuales (`#@param`). Solo eliges el modelo deseado y pulsas Play.

## Parte 1: Instalacion y Configuracion del Entorno
Monta Google Drive, configura credenciales de Kaggle, clona/actualiza el repositorio e instala dependencias.

In [ ]:
#@title 1. Inicializar Entorno BellAI { display-mode: "form" }
import os
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Token de Kaggle
KAGGLE_API_TOKEN = 'TU_KAGGLE_API_TOKEN_AQUI' #@param {type:"string"}
if KAGGLE_API_TOKEN and KAGGLE_API_TOKEN != 'TU_KAGGLE_API_TOKEN_AQUI':
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
    !mkdir -p ~/.kaggle && echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
    print('[*] Kaggle API Token configurado exitosamente.')

# 3. Control de clonado / actualizacion del repositorio
actualizar_con_git_pull = True #@param {type:"boolean"}
if not os.path.exists('/content/modelIATrainer'):
    print('[*] Repositorio no encontrado. Clonando por primera vez...')
    !git clone https://github.com/ldgd2/modelIATrainer.git
    %cd /content/modelIATrainer
    !pip install -q -r requirements.txt
else:
    print('[*] Repositorio ya detectado en /content/modelIATrainer. Omitiendo clonado.')
    %cd /content/modelIATrainer
    if actualizar_con_git_pull:
        print('[*] Verificando ultimos cambios con git pull...')
        !git pull
    else:
        print('[*] Omitiendo actualizacion con git pull.')

print('\n' + '='*55)
print(' [*] ENTORNO BELLAI LISTO PARA ENTRENAR')
print('='*55)


## Parte 2: Modulo 1 - Armario Digital
Deteccion y segmentacion de prendas con DeepFashion2 hacia `wardrobe_v1.pt`.

In [ ]:
#@title Entrenar Armario Digital { display-mode: "form" }
%cd /content/modelIATrainer

epocas = 10 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
arquitectura = "mobilenet_v3_large (Recomendado para Movil: Rapido y Ligero)" #@param ["mobilenet_v3_large (Recomendado para Movil: Rapido y Ligero)", "mobilenet_v3_small (Ultra-ligero Movil)", "resnet34 (Alta precision GPU)", "resnet50 (Maximo rendimiento Servidor)"]
fuente_datos = "Auto (Usar local si existe / Descargar si falta)" #@param ["Auto (Usar local si existe / Descargar si falta)", "Descargar con Kagglehub", "Enlaces directos wget", "Simulacion / Verificacion"]
kaggle_slug = "khyeh0719/deepfashion" #@param {type:"string"}
link_train = "" #@param {type:"string"} 
link_val = "" #@param {type:"string"}
password = "2019Deepfashion2**" #@param {type:"string"}

continuar_desde_ultimo_checkpoint = True #@param {type:"boolean"}
checkpoint_manual = "" #@param {type:"string"}

import os

# 1. Seleccion de arquitectura (Compatible con movil y GPU)
if "50" in arquitectura:
    arg_backbone = '--backbone resnet50'
elif "34" in arquitectura:
    arg_backbone = '--backbone resnet34'
elif "small" in arquitectura:
    arg_backbone = '--backbone mobilenet_v3_small'
else:
    arg_backbone = '--backbone mobilenet_v3_large'

# 2. Gestion de datos segun condicion
args_datos = ""
dataset_train_path = "/content/temp_armario/dataset/train"
tiene_datos_locales = os.path.exists(dataset_train_path) and len(os.listdir(dataset_train_path)) > 0

if fuente_datos == "Auto (Usar local si existe / Descargar si falta)":
    if tiene_datos_locales:
        print("[*] Dataset local detectado en /content/temp_armario/dataset/train. Omitiendo descarga.")
        args_datos = ""
    elif link_train:
        print("[*] Dataset no encontrado localmente. Utilizando enlaces directos...")
        args_datos = f'--link_train "{link_train}"'
        if link_val:
            args_datos += f' --link_val "{link_val}"'
    else:
        print("[*] Dataset no encontrado localmente. Utilizando Kagglehub...")
        args_datos = f'--download_data --kaggle_slug "{kaggle_slug}"'
elif fuente_datos == "Descargar con Kagglehub":
    args_datos = f'--download_data --kaggle_slug "{kaggle_slug}"'
elif fuente_datos == "Enlaces directos wget":
    if link_train:
        args_datos += f' --link_train "{link_train}"'
    if link_val:
        args_datos += f' --link_val "{link_val}"'

# 3. Gestion de continuacion y checkpoints automaticos
if continuar_desde_ultimo_checkpoint:
    args_resume = '--resume "auto"'
    print("[*] Modo reanudacion: Se detectara automaticamente el ultimo checkpoint guardado en Drive.")
elif checkpoint_manual:
    args_resume = f'--resume "{checkpoint_manual}"'
    print(f"[*] Modo reanudacion: Checkpoint manual especificado: {checkpoint_manual}")
else:
    args_resume = ""
    print("[*] Modo nuevo: Se iniciara el entrenamiento desde la epoca 1.")

cmd = f'python train.py --module armario_digital --epochs {epocas} --save_every 1 --batch_size {batch_size} {arg_backbone} --password "{password}" {args_datos} {args_resume}'.strip()
print(f'\n[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 3: Modulo 2 - Diagnostico Morfologico
Analisis de rostro, silueta corporal y cabello (FairFace, COCO Keypoints, CelebAMask-HQ).

In [ ]:
#@title Entrenar Diagnostico Morfologico { display-mode: "form" }
%cd /content/modelIATrainer

subtarea = "colorimetria" #@param ["colorimetria", "morfologia", "capilar"]
epocas = 10 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
arquitectura = "mobilenet_v3_large (Recomendado para Movil: Rapido y Ligero)" #@param ["mobilenet_v3_large (Recomendado para Movil: Rapido y Ligero)", "mobilenet_v3_small (Ultra-ligero Movil)", "resnet34 (Alta precision GPU)", "resnet50 (Maximo rendimiento Servidor)"]
fuente_datos = "Auto (Usar local si existe / Descargar si falta)" #@param ["Auto (Usar local si existe / Descargar si falta)", "Descargar con Kagglehub", "Simulacion / Verificacion"]

continuar_desde_ultimo_checkpoint = True #@param {type:"boolean"}
checkpoint_manual = "" #@param {type:"string"}

import os

# 1. Seleccion de arquitectura (Compatible con movil y GPU)
if "50" in arquitectura:
    arg_backbone = '--backbone resnet50'
elif "34" in arquitectura:
    arg_backbone = '--backbone resnet34'
elif "small" in arquitectura:
    arg_backbone = '--backbone mobilenet_v3_small'
else:
    arg_backbone = '--backbone mobilenet_v3_large'

# 2. Gestion de datos
local_task_data = f"/content/temp_diagnostico/dataset/{subtarea}"
tiene_datos_locales = os.path.exists(local_task_data) and len(os.listdir(local_task_data)) > 0

if fuente_datos == "Auto (Usar local si existe / Descargar si falta)":
    if tiene_datos_locales:
        print(f"[*] Dataset local detectado para '{subtarea}'. Omitiendo descarga.")
        dl_flag = ""
    else:
        print(f"[*] Dataset no encontrado localmente para '{subtarea}'. Descargando...")
        dl_flag = "--download_data"
elif fuente_datos == "Descargar con Kagglehub":
    dl_flag = "--download_data"
else:
    dl_flag = ""

# 3. Gestion de checkpoints
if continuar_desde_ultimo_checkpoint:
    args_resume = '--resume "auto"'
    print("[*] Modo reanudacion: Se detectara automaticamente el ultimo checkpoint guardado en Drive.")
elif checkpoint_manual:
    args_resume = f'--resume "{checkpoint_manual}"'
    print(f"[*] Modo reanudacion: Checkpoint manual especificado: {checkpoint_manual}")
else:
    args_resume = ""
    print("[*] Modo nuevo: Se iniciara el entrenamiento desde la epoca 1.")

cmd = f'python train.py --module diagnostico_morfologico --task {subtarea} --epochs {epocas} --save_every 1 --batch_size {batch_size} {arg_backbone} {dl_flag} {args_resume}'.strip()
print(f'\n[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 4: Modulo 3 - Motor de Recomendacion
Compatibilidad estetica de conjuntos de ropa (PolyVore -> `style_recommender_v1.pt`).

In [ ]:
#@title Entrenar Motor de Recomendacion { display-mode: "form" }
%cd /content/modelIATrainer

epocas = 10 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
arquitectura = "mobilenet_v3_large (Recomendado para Movil: Rapido y Ligero)" #@param ["mobilenet_v3_large (Recomendado para Movil: Rapido y Ligero)", "mobilenet_v3_small (Ultra-ligero Movil)", "resnet34 (Alta precision GPU)", "resnet50 (Maximo rendimiento Servidor)"]
fuente_datos = "Auto (Descargar Kagglehub / Usar local si existe)" #@param ["Auto (Descargar Kagglehub / Usar local si existe)", "Descargar con Kagglehub (Re-PolyVore)", "Descargar URL SharePoint", "Simulacion / Verificacion"]
kaggle_slug = "nastarang/re-polyvore" #@param {type:"string"}
link_polyvore = "" #@param {type:"string"}

continuar_desde_ultimo_checkpoint = True #@param {type:"boolean"}
checkpoint_manual = "" #@param {type:"string"}

import os

# 1. Seleccion de arquitectura (Compatible con movil y GPU)
if "50" in arquitectura:
    arg_backbone = '--backbone resnet50'
elif "34" in arquitectura:
    arg_backbone = '--backbone resnet34'
elif "small" in arquitectura:
    arg_backbone = '--backbone mobilenet_v3_small'
else:
    arg_backbone = '--backbone mobilenet_v3_large'

# 2. Gestion de datos
local_poly_data = "/content/temp_recomendacion/dataset/polyvore"
tiene_datos_locales = os.path.exists(local_poly_data) and len(os.listdir(local_poly_data)) > 0

if fuente_datos == "Auto (Descargar Kagglehub / Usar local si existe)":
    if tiene_datos_locales:
        print("[*] Dataset local PolyVore detectado. Omitiendo descarga.")
        args_data = ""
    else:
        print("[*] Descargando Re-PolyVore via Kagglehub...")
        args_data = f'--download_data --kaggle_slug "{kaggle_slug}"'
elif fuente_datos == "Descargar con Kagglehub (Re-PolyVore)":
    args_data = f'--download_data --kaggle_slug "{kaggle_slug}"'
elif fuente_datos == "Descargar URL SharePoint":
    args_data = f'--link_polyvore "{link_polyvore}"' if link_polyvore else ""
else:
    args_data = ""

# 3. Gestion de checkpoints
if continuar_desde_ultimo_checkpoint:
    args_resume = '--resume "auto"'
    print("[*] Modo reanudacion: Se detectara automaticamente el ultimo checkpoint guardado en Drive.")
elif checkpoint_manual:
    args_resume = f'--resume "{checkpoint_manual}"'
    print(f"[*] Modo reanudacion: Checkpoint manual especificado: {checkpoint_manual}")
else:
    args_resume = ""
    print("[*] Modo nuevo: Se iniciara el entrenamiento desde la epoca 1.")

cmd = f'python train.py --module motor_recomendacion --epochs {epocas} --save_every 1 --batch_size {batch_size} {arg_backbone} {args_data} {args_resume}'.strip()
print(f'\n[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 5: Modulo 4 - Simulacion Virtual
Probador virtual y mapeo de maquillaje (VITON-HD, MT-Dataset).

In [ ]:
#@title Entrenar Simulacion Virtual { display-mode: "form" }
%cd /content/modelIATrainer

subtarea = "probador" #@param ["probador", "maquillaje"]
epocas = 10 #@param {type:"integer"}
batch_size = 16 #@param {type:"integer"}
fuente_datos = "Auto (Usar local si existe / Descargar si falta)" #@param ["Auto (Usar local si existe / Descargar si falta)", "Descargar con Kagglehub", "Simulacion / Verificacion"]

continuar_desde_ultimo_checkpoint = True #@param {type:"boolean"}
checkpoint_manual = "" #@param {type:"string"}

import os

# 1. Gestion de datos
local_sim_data = f"/content/temp_simulacion/dataset/{subtarea}"
tiene_datos_locales = os.path.exists(local_sim_data) and len(os.listdir(local_sim_data)) > 0

if fuente_datos == "Auto (Usar local si existe / Descargar si falta)":
    if tiene_datos_locales:
        print(f"[*] Dataset local detectado para '{subtarea}'. Omitiendo descarga.")
        dl_flag = ""
    else:
        print(f"[*] Dataset no encontrado localmente para '{subtarea}'. Descargando...")
        dl_flag = "--download_data"
elif fuente_datos == "Descargar con Kagglehub":
    dl_flag = "--download_data"
else:
    dl_flag = ""

# 2. Gestion de checkpoints
if continuar_desde_ultimo_checkpoint:
    args_resume = '--resume "auto"'
    print("[*] Modo reanudacion: Se detectara automaticamente el ultimo checkpoint guardado en Drive.")
elif checkpoint_manual:
    args_resume = f'--resume "{checkpoint_manual}"'
    print(f"[*] Modo reanudacion: Checkpoint manual especificado: {checkpoint_manual}")
else:
    args_resume = ""
    print("[*] Modo nuevo: Se iniciara el entrenamiento desde la epoca 1.")

cmd = f'python train.py --module simulacion_virtual --task {subtarea} --epochs {epocas} --save_every 1 --batch_size {batch_size} {dl_flag} {args_resume}'.strip()
print(f'\n[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 6: Modulo 5 - Asistente Conversacional
Descarga y empaquetado del modelo LLM local en formato GGUF desde Hugging Face.

In [ ]:
#@title Entrenar / Descargar Asistente Conversacional { display-mode: "form" }
%cd /content/modelIATrainer

modo = "Entrenar Asistente Estilista BellAI (Fine-Tuning)" #@param ["Entrenar Asistente Estilista BellAI (Fine-Tuning)", "Solo descargar modelo GGUF base"]
modelo_base = "TinyLlama-1.1B (Compatible Movil)" #@param ["TinyLlama-1.1B (Compatible Movil)", "Qwen2.5-1.5B (Alta Calidad)", "Llama-3.2-1B (Meta)"]
epocas = 3 #@param {type:"integer"}
batch_size = 4 #@param {type:"integer"}
continuar_desde_ultimo_checkpoint = True #@param {type:"boolean"}
checkpoint_manual = "" #@param {type:"string"}

repo_id = "TheBloke/Llama-2-7B-Chat-GGUF" #@param {type:"string"}
filename = "bellai_assistant_8b.gguf" #@param {type:"string"}

import os

if modo == "Entrenar Asistente Estilista BellAI (Fine-Tuning)":
    if "Qwen" in modelo_base:
        base_slug = "Qwen/Qwen2.5-1.5B-Instruct"
    elif "Llama-3.2" in modelo_base:
        base_slug = "meta-llama/Llama-3.2-1B-Instruct"
    else:
        base_slug = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    if continuar_desde_ultimo_checkpoint:
        args_resume = '--resume "auto"'
        print("[*] Modo reanudacion: Se detectara automaticamente el ultimo checkpoint guardado en Drive.")
    elif checkpoint_manual:
        args_resume = f'--resume "{checkpoint_manual}"'
        print(f"[*] Modo reanudacion: Checkpoint manual especificado: {checkpoint_manual}")
    else:
        args_resume = ""
        print("[*] Modo nuevo: Se iniciara el fine-tuning desde la epoca 1.")

    cmd = f'python train.py --module asistente_conversacional --mode train --base_model "{base_slug}" --epochs {epocas} --batch_size {batch_size} {args_resume}'.strip()
else:
    cmd = f'python train.py --module asistente_conversacional --mode download --repo_id "{repo_id}" --filename "{filename}"'.strip()

print(f'\n[*] Ejecutando: {cmd}\n')
!{cmd}
